# Week 1: Data Quality and Knowledge Discovery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/advanced-data-mining-course/blob/main/weeks/01/Week_01_Data_Quality_Knowledge_Discovery.ipynb)

**Advanced Data Mining · Master level · 180-minute core session**

Last course, you learned to build and inspect representations. This course asks what knowledge we can responsibly extract from data. Our first question is: **would you trust this finding if the cleaning policy changed?**

## Learning goals

By the end, you can distinguish invoice lines, invoices, and customers; investigate completeness, validity, uniqueness, and consistency; explain the difference between a suspicious record and a confirmed error; and show how an analysis policy changes a conclusion. You will support one claim with a table, a plot, and a limitation.

## How to work through this notebook

This is a small chapter and a laboratory notebook in one. Read each substantial block independently before running its experiments. You do not need to understand every line of Python before beginning: follow the explanation of what the calculation means, run the baseline, and then change one decision.

| Activity | Time |
|---|---:|
| Opening and environment setup | 10 min |
| Reading A and its written checkpoint | 15 min |
| Tiny-data experiment and short discussion | 15 min |
| Reading B and its written checkpoint | 15 min |
| Load and inspect the real data | 10 min |
| Independent investigation 1, including policy note | 40 min |
| Compare reasoning with a partner | 5 min |
| Reading C and its written checkpoint | 15 min |
| Run the distribution examples | 10 min |
| Independent investigation 2, including evidence report | 40 min |
| Whole-class synthesis | 5 min |
| **Total, excluding breaks** | **180 min** |

The 10–15-minute reading windows include thinking, annotating, and answering the checkpoint, not just reading speed. Mark one sentence you understand well and one you would like to discuss. Self-check answers are folded away; attempt your explanation before opening them.

**Course habit:** question → inspect → state assumptions → change one decision → compare → explain.

Work alone first and compare reasoning afterwards. If you finish early, deepen your investigation using its extension rather than waiting for the class. These activities prepare you for Assignment 1; they are not additional graded submissions.

## Environment and getting started

**Colab:** click **Open in Colab** above, connect to a CPU runtime, and choose **Runtime → Run all**. The setup cell installs the required packages; the data cell downloads and verifies the small course dataset automatically. No upload, ZIP extraction, account token, or manual data preparation is required. If Colab asks for a runtime restart after installation, restart and choose Run all again.

**Local:** from the course root, run `uv sync --locked`, then `uv run jupyter lab`. Open this notebook with the environment's Python kernel and run cells in order. The supplied `data/` file is used locally; otherwise the same download runs automatically. First-time installation/download needs internet. Afterward, local cached data is reused.

**Prerequisites:** Python variables, pandas tables, basic plots, and the previous course's idea that a representation determines what information is available. CPU is sufficient; no neural-network training is required today.

Run the baseline first. During the investigations, change one setting and rerun its relevant cells. Restart and run all if changes leave you unsure which variables are current.

In [ ]:
# Install automatically in Colab; local environments are managed with uv.
import sys
import subprocess
import importlib.util

if importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        'numpy>=2,<3', 'pandas>=2.2,<3', 'matplotlib>=3.9,<4',
        'scikit-learn>=1.6,<1.8'
    ])
    print('Colab dependencies ready. Continue with the next cell.')
else:
    print('Local runtime: using the course environment (uv sync --locked).')

In [ ]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
pd.set_option('display.max_columns', 12)
plt.rcParams.update({'figure.figsize': (8, 4), 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})
print('NumPy:', np.__version__, '| pandas:', pd.__version__)

## Reading A — What does a row allow us to claim?

**Read and reflect for 10–15 minutes.** Read through “Checkpoint A” before running the tiny example. Your aim is to explain why a technically correct calculation can still answer the wrong question.

### A number needs a question

Imagine that a retailer asks: “Which country buys the most?” You open a table, count the rows by country, and report the largest count. The code works. But what did you count? If each row records a product on an invoice, a customer buying ten different products may contribute ten rows. A customer buying one hundred units of the same product may contribute only one. Counting rows answers a question about recorded product lines, not necessarily purchases, people, or money.

Before calculating, translate the vague request into a sentence you could defend: “In this extract, which country has the largest sum of positive recorded sales value under the stated filtering policy?” This is less catchy than “best market,” but it tells another person what the result means. It also reveals the decisions that need to be made: which records count, how value is calculated, and which population is being described.

This connection between a question and a calculation is central to data mining. Later we will discover clusters and association rules. Those algorithms can find convincing patterns in data that represents the wrong question. Learning to specify the question now makes the later methods more useful.

### Rows, invoices, and customers are different units

The **observation unit** is the thing represented by one observation. In our source table it is an invoice line: a recorded product, quantity, and price on an invoice. To discuss a purchase, we often combine lines belonging to the same invoice. To discuss purchasing behavior, we may combine invoices belonging to a customer. Each aggregation changes what one observation means.

Consider this invented example. Customer Ana places invoice A containing three product lines worth £10, £20, and £30. Customer Ben places invoice B containing one line worth £40. The table has four lines, two invoices, and two customers. Mean line value is £25: £100 divided by four. Mean invoice value is £50: £100 divided by two. Neither answer is arithmetically wrong. They describe different units.

Now suppose Ana places another invoice next week. Counting distinct invoice IDs would increase the number of purchases. Counting distinct customer IDs would still give two people in this invented story. In real data, an ID identifies a recorded account or entity; we would need documentation before asserting that every ID corresponds to exactly one person.

A **key** identifies the chosen unit. `InvoiceNo` can identify an invoice, but repeats legitimately in an invoice-line table. `CustomerID` also repeats legitimately across a customer's purchases. Deleting every repeated invoice ID would remove real product lines. Even two entirely identical rows are not automatically errors: they might be duplicated exports or separate legitimate records whose distinguishing field was never supplied.

### Quality depends on the intended use

We will inspect four useful dimensions. **Completeness** asks whether needed values are present. A missing customer ID prevents us from attaching that record to a known customer's history. It does not necessarily prevent us from adding the record's quantity times price to a country total. “Missing” and “unusable” are therefore different judgments.

**Validity** asks whether a value meets a defined rule. A negative quantity violates a rule for positive purchases, but may correctly record a return. **Uniqueness** asks whether the same observation has been represented more than once, relative to an appropriate identifier. **Consistency** asks whether fields and conventions agree—for example, whether `UK` and `uk` mean the same category, or whether a cancellation convention agrees with the recorded signs.

These checks do not establish factual accuracy. A mistyped quantity of 8 instead of 3 might satisfy every rule. An unusual quantity of 1,000 could be accurate for a wholesale order. Accuracy often requires external evidence such as a source document, system specification, or clarification from the data owner. An audit produces questions and evidence, not an automatic list of rows to delete.

### Worked example: a return changes the question

Suppose an invoice records two items at £5 each. Its line value is £10. A subsequent return records quantity −1 at £5, giving −£5. Adding both values produces a signed total of £5. Adding only the positive purchase produces £10. The difference is not a calculation mistake; one summary includes the recorded return and the other describes positive sales only.

Taking the absolute value of the negative quantity would produce £15 in total and incorrectly turn the return into another sale. A transformation that makes a table look cleaner can make its meaning worse. We will preserve the original records and create explicitly named views for different questions.

### Checkpoint A — explain before running

Write two or three sentences in your own words: What is the difference between an invoice line and an invoice? When might you retain a row with no customer ID? What additional evidence would you need before removing an exact repeated row?

<details><summary>Self-check — open after writing</summary>

An invoice can contain several product lines, so a count of lines is not a count of purchases. A row without customer identity may still contribute to an analysis of recorded sales by country, provided the fields required for that question are present. Repetition alone does not prove an export error; a source line identifier or confirmation of the export process would help distinguish duplicates from legitimate repeated events.

</details>

## Small experiment — a dataset where we know the story

The following table is deliberately synthetic. Here the data owner confirms that the repeated `A` row is an export duplicate, `uk` means `UK`, and `C1` is a genuine return. A missing customer ID remains unknown. This evidence justifies choices in this example; it is not evidence about our later real dataset.

Spend about 10 minutes predicting and running the baseline, then 5 minutes comparing your explanation with a partner. First calculate the value of each line on paper. Then predict how removing the confirmed repeat and excluding returns will affect the total. Run the cells to check your reasoning.

For TODO 2, change only the unusually large quantity. Record your prediction before rerunning the two cells. Restore the original quantity afterwards. You have finished when you can explain why a changed total is expected without calling every unusual observation an error.

In [ ]:
toy = pd.DataFrame({
    'invoice': ['A', 'A', 'B', 'C1', 'D', 'E'],
    'customer': ['u1', 'u1', None, 'u1', 'u2', 'u3'],
    'country': ['UK', 'UK', 'uk', 'UK', 'DE', 'DE'],
    'quantity': [2, 2, 3, -1, 1, 1000],
    'price': [5., 5., 4., 5., 0., 2.]})
display(toy)
display(pd.Series({'missing_customer': toy.customer.isna().sum(),
                   'exact_repeats_after_first': toy.duplicated().sum(),
                   'nonpositive_price': (toy.price <= 0).sum(),
                   'negative_quantity': (toy.quantity < 0).sum()}))

In [ ]:
# TODO 1: Predict the three totals before running. Explain each difference.
toy['value'] = toy.quantity * toy.price
toy_unique = toy.drop_duplicates().copy()
toy_unique['country'] = toy_unique.country.str.upper()
toy_sales = toy_unique[(toy_unique.quantity > 0) & (toy_unique.price > 0)]
display(pd.Series({'raw_signed_value': toy.value.sum(),
                   'deduplicated_signed_value': toy_unique.value.sum(),
                   'positive_sales_value': toy_sales.value.sum()}))
# TODO 2: Change the 1000 quantity to 10 in the original toy cell and rerun.
# Would an unusually large order alone justify deleting it? Restore 1000 afterwards.

### Interpreting the tiny example

The signed line value is $v_i=q_i p_i$. Summing it includes negative quantities; summing only positive sales changes the estimand—the quantity we intend to describe. Neither sum is automatically accounting revenue: taxes, shipping, refunds, and other conventions may matter.

**Common confusion:** a return is not an invalid sale to “repair” by taking the absolute quantity. That would invent a purchase. Keep the original table and build named views for specific questions.

Write one sentence: “For question ___, I retain/exclude ___ because ___.” The justification is more important than the number of rows removed.

## Reading B — An audit is evidence; cleaning is a decision

**Read and reflect for 10–15 minutes.** This block prepares you to investigate the real extract without waiting for a line-by-line demonstration. Complete Checkpoint B, then load the data and run the audit.

### Start with what the file actually contains

A **schema** describes the fields and how they are represented: strings, numbers, dates, and so on. A schema check is useful because a program may accept a value while assigning it the wrong meaning. A customer identifier that contains digits should still be treated as an identifier. Averaging customer IDs would produce a number, but no useful description of customers.

Our loader keeps identifiers as strings and parses invoice dates as dates. The checksum verifies that everyone uses the same supplied file; it does not certify that the original records are true. When a notebook reports an integrity error, obtain the supplied file again rather than disabling the check. Perform experiments on DataFrame copies so the original file remains available for comparison.

The first audit table reports missing values by column. In pandas, `isna()` produces a Boolean table: each entry is `True` where a value is missing. Summing a Boolean column counts its `True` values. Taking its mean gives the fraction that is `True`, because these operations treat `True` as one and `False` as zero. Multiplying by 100 expresses that fraction as a percentage. This is why a short line of code can produce both counts and missingness rates.

### Why we inspect records behind the flags

An **audit flag** is a condition that marks a record for attention. It is not a verdict. A flag for nonpositive quantity might identify a return, an adjustment, or a source-system error. To decide what to do, inspect examples and read the field definitions. A useful audit note states the observed fact, a possible explanation, and what evidence would distinguish explanations.

For example: “These rows have negative quantities; some invoice IDs begin with C; the source defines that prefix as cancellation. I will exclude cancellation records when describing positive purchases, while retaining the raw records for a separate return analysis.” This reasoning is stronger than “negative values are bad.” It connects the rule to the question and makes the decision reproducible.

Flags can overlap. Imagine ten rows: three lack customer IDs, two have negative quantities, and one row belongs to both sets. Adding three and two counts that row twice. There are four distinct flagged rows, not five. In our code, `flags.any(axis=1)` asks whether each row has at least one flag. By contrast, `flags.sum()` counts each condition separately. Both results are useful when named correctly.

### A named view makes assumptions visible

We use **raw** for the unmodified source table and **sales** for a view answering one specific question. A view is a derived table, not a repaired version of reality. Our positive-sales view excludes cancellation invoices and requires positive quantity and price. It optionally drops exact repeats and optionally requires customer identity.

The `sales_view` function puts those choices in one place. Its input `frame` is the table to inspect; `drop_repeats` and `require_customer` are switches. It returns a filtered table and a record of how many rows remain after each step. You can call it twice with one switch changed and compare outcomes while holding everything else fixed.

This is a **sensitivity analysis**: ask how a result responds to a reasonable alternative assumption. If a policy changes the result very little, that particular conclusion may be insensitive to that choice. It does not mean every conclusion is robust. An unchanged overall total can hide a large change for one country or a small customer group. If a result changes substantially, report the dependence instead of selecting the most convenient answer.

### Counts, denominators, and fair comparisons

The function also produces a sequential row waterfall. Suppose 100 rows become 95 after removing repeats, then 88 after a cancellation filter. The recorded removals are five and seven; their sum describes the twelve rows removed along that sequence. Some cancellation rows might already have been removed as repeats. Therefore the attribution to individual steps depends on their order. The audit flag table answers a different question: how many source rows satisfy each condition before filtering?

For numerical comparisons, always name a baseline. If policy A gives £100 and policy B gives £90, the change from A to B is −£10, or −10% of A. The change from B to A is +£10, or about +11.1% of B. A percentage without its denominator is incomplete. To isolate the effect of requiring identity, compare two policies with the same duplicate policy; comparing policies that differ in both switches measures their combined effect.

### Missingness and sampling limit our claims

Deleting missing values is not automatically harmless. Suppose guest checkout makes a customer ID less likely to be recorded. Requiring identity could then emphasize registered customers. That is a possible explanation, not something our extract proves. Conversely, filling every missing ID with `unknown` and grouping by that value creates one artificial customer with many unrelated purchases.

Our extract deliberately samples known customer histories and anonymous invoices differently. It is useful for learning the consequences of an ID requirement, but its missing-ID fraction is not an estimate for the full retailer. Keeping all observed rows for sampled customers helps preserve their history within the source window; it does not make every invoice equally likely to enter the extract or reveal purchases outside that window.

### Checkpoint B — choose a comparison

Write down which two policy settings would isolate the effect of dropping exact repeats. Then explain why five missing-ID flags plus three cancellation flags need not mean eight distinct affected rows. Finally, state one question that this sample cannot answer about the retailer as a whole.

<details><summary>Self-check — open after attempting the questions</summary>

Hold the identity requirement fixed and change only `drop_repeats`. Flag counts can overlap on the same rows, so distinct affected rows require a union of the flags. For example, the extract cannot directly estimate the retailer's overall missing-ID rate or total sales, because its selection mechanisms and coverage do not support those claims.

</details>

### Our real dataset: a reproducible retail extract

We use a supplied extract of [UCI Online Retail](https://archive.ics.uci.edu/dataset/352/online+retail), a UK online retailer's historical invoice lines. A row is a **product line**, not a customer and not necessarily an entire purchase. Prices are in pounds sterling; invoice IDs beginning with `C` mark cancellations.

The extract preserves every source row for 400 sampled customer IDs, plus rows from ten sampled anonymous invoices per calendar month. It retains source defects and cancellations. Known and anonymous records have different sampling mechanisms: their relative proportions cannot estimate missing-ID prevalence in the original population. This is a historical case study, not a claim about present-day shoppers. December 2011 is incomplete.

**Attribution:** Chen, D. (2015), *Online Retail*, UCI, [DOI: 10.24432/C5BW33](https://doi.org/10.24432/C5BW33), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). The course adaptation is customer/invoice selection plus CSV conversion; see `data/README.md` and `retail_manifest.json` for provenance. No synthetic defects were added to the real extract.

In [ ]:
# Prefer the bundled/local cache; otherwise fetch the public course dataset.
import urllib.request
import urllib.error

filename = 'online_retail_teaching.csv.gz'
DATA_URL = 'https://raw.githubusercontent.com/rrfhwn/advanced-data-mining-course/main/data/online_retail_teaching.csv.gz'
expected_sha256 = '8503a734cd203ed63df3292c2a12d0b803206dc18ed67a7dc83664fd307d96d8'
candidates = [Path.cwd() / filename]
for folder in [Path.cwd(), *Path.cwd().parents]:
    candidates.append(folder / 'data' / filename)
data_path = next((p for p in candidates if p.is_file()), None)

if data_path is None:
    data_path = Path.cwd() / 'data' / filename
    print('Downloading the course dataset (approximately 610 KB)...')
    try:
        with urllib.request.urlopen(DATA_URL, timeout=60) as response:
            payload = response.read()
    except (urllib.error.URLError, TimeoutError) as error:
        raise RuntimeError(
            'Could not download the course dataset. Check your internet connection and retry. '
            'If the URL is unavailable, ask the instructor to check the public course release. '
            f'URL: {DATA_URL}'
        ) from error
    if hashlib.sha256(payload).hexdigest() != expected_sha256:
        raise ValueError('Downloaded data does not match this notebook version. Reopen the current course notebook.')
    data_path.parent.mkdir(parents=True, exist_ok=True)
    data_path.write_bytes(payload)
else:
    print('Using the cached course dataset.')

if hashlib.sha256(data_path.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError('Cached dataset differs from this notebook version. Use a fresh Colab runtime or restore the matching course data locally.')
raw = pd.read_csv(data_path, dtype={'InvoiceNo': 'string', 'StockCode': 'string',
                                  'CustomerID': 'string'}, parse_dates=['InvoiceDate'])
assert len(raw) > 0 and raw.InvoiceDate.notna().all()
print(f'{len(raw):,} rows; {raw.CustomerID.nunique()} observed customer IDs')
display(raw.head(4))

## 3. Audit before cleaning

Start with a schema, missingness, and explicit checks. IDs are strings even when they look numeric; arithmetic on a customer identifier has no useful interpretation. `Quantity` and `UnitPrice` are measurements with different units.

Flags may overlap: a repeated row may also lack an ID. Their counts must not be added and presented as a total number of bad rows. We will inspect flags before choosing a policy.

**Pause and predict:** which missing fields prevent a country-level sales analysis? Which prevent a customer-level analysis? Why are those different?

In [ ]:
display(pd.DataFrame({'dtype': raw.dtypes.astype(str),
                      'missing_n': raw.isna().sum(),
                      'missing_percent': 100 * raw.isna().mean()}).round(2))
flags = pd.DataFrame({
    'missing_customer': raw.CustomerID.isna(),
    'missing_description': raw.Description.isna(),
    'exact_repeat': raw.duplicated(),
    'cancellation': raw.InvoiceNo.str.upper().str.startswith('C', na=False),
    'nonpositive_quantity': raw.Quantity <= 0,
    'nonpositive_price': raw.UnitPrice <= 0})
display(flags.sum().sort_values(ascending=False).rename('flagged_rows'))
display(raw.loc[flags.any(axis=1)].head(6))

In [ ]:
# TODO 3: Choose another flag; inspect examples before assigning a meaning.
flag_to_inspect = 'nonpositive_quantity'
display(raw.loc[flags[flag_to_inspect],
                ['InvoiceNo', 'Description', 'Quantity', 'UnitPrice', 'CustomerID']].head(8))
flags.mean().mul(100).sort_values().plot.barh(color='#277da8')
plt.xlabel('Percent of extract rows (flags overlap)')
plt.title('Audit findings are questions, not deletion instructions')
plt.tight_layout(); plt.show()

## Independent investigation 1 — compare policies (40 minutes)

Begin with the question: **how much positive recorded sales value is present in this extract?** The baseline excludes cancellations and nonpositive quantity/price, drops exact repeats as a stated assumption, and retains anonymous sales. Read the function once, run it, and inspect its row waterfall. You do not need to rewrite it.

Use roughly 8 minutes for the baseline, 20 minutes for your investigation, and 12 minutes for the policy note. Choose one route below. Each route requires a baseline, one controlled change, a numerical comparison, and an interpretation. The other route is an extension if you finish early.

**Route A: how much does the repeat policy matter?** Compare `drop_repeats=False` and `True`, holding `require_customer=False`. Report row counts, invoice counts, positive value, and absolute and percentage differences. Then inspect one repeated record. Explain why a reduction in line count need not reduce the number of invoices.

**Route B: who disappears when identity is required?** Hold `drop_repeats=True` and compare `require_customer=False` and `True`. Report the same quantities, then inspect one excluded anonymous invoice. Explain why you would retain or exclude it for two different questions: positive sales by country and customer purchasing histories.

Before running your changed policy, predict the direction of the change and explain why. Keep `raw` untouched. Save your alternative to a new variable rather than replacing the baseline `sales`, which the later examples reuse.

<details><summary>Hints if you are stuck</summary>

Call `sales_view(raw, drop_repeats=False, require_customer=False)` for the retain-repeats view. It returns two objects, so use `alternative, alternative_audit = ...`. `len(alternative)` counts rows, `alternative.InvoiceNo.nunique()` counts distinct invoices, and `alternative.LineValue.sum()` adds positive recorded value. For the identity route, `raw.CustomerID.isna()` locates anonymous records. To inspect repeats beyond their first occurrence, use `raw.loc[raw.duplicated()]`. Percentage change is `100 * (changed_total - baseline_total) / baseline_total`.

</details>

**Finished?** You can identify exactly which policy switch changed, reproduce both totals, and explain one limitation. A screenshot of a result without a stated question is not yet a finished investigation.

In [ ]:
def sales_view(frame, drop_repeats=True, require_customer=False):
    view = frame.copy()
    audit = [('input', len(view))]
    if drop_repeats:
        view = view.drop_duplicates()
        audit.append(('after exact-repeat policy', len(view)))
    view = view[~view.InvoiceNo.str.upper().str.startswith('C', na=False)]
    audit.append(('after cancellation filter', len(view)))
    view = view[(view.Quantity > 0) & (view.UnitPrice > 0)]
    audit.append(('after positive quantity/price', len(view)))
    if require_customer:
        view = view[view.CustomerID.notna()]
        audit.append(('after known-ID requirement', len(view)))
    view = view.assign(LineValue=view.Quantity * view.UnitPrice)
    return view, pd.DataFrame(audit, columns=['stage', 'rows'])

sales, waterfall = sales_view(raw)
waterfall['removed_at_step'] = waterfall.rows.shift(1).sub(waterfall.rows).fillna(0).astype(int)
display(waterfall)
assert (sales.Quantity > 0).all() and (sales.UnitPrice > 0).all()

In [ ]:
policies = []
for name, drop_repeats, require_customer in [
    ('positive sales / retain repeats', False, False),
    ('positive sales / drop repeats', True, False),
    ('positive sales / known IDs only', True, True)]:
    view, _ = sales_view(raw, drop_repeats, require_customer)
    policies.append({'policy': name, 'rows': len(view),
                     'invoices': view.InvoiceNo.nunique(),
                     'positive_value_GBP': view.LineValue.sum()})
policy_summary = pd.DataFrame(policies).set_index('policy')
display(policy_summary.round(2))
# TODO 4: Calculate the absolute and percentage difference for two policies.
baseline_value = policy_summary.loc['positive sales / retain repeats', 'positive_value_GBP']
policy_summary['difference_percent'] = 100 * (policy_summary.positive_value_GBP / baseline_value - 1)
display(policy_summary[['difference_percent']].round(2))

### Your policy note — write in this cell

**TODO 5:** write 150–200 words, using the prompts below as a scaffold. Include actual values from your run, not only a statement that they “changed.”

- My question and observation unit: …
- Baseline policy and one controlled change: …
- Baseline total, changed total, absolute difference, and percentage denominator: …
- A row I inspected and what it can or cannot tell me: …
- My interpretation, an uncertain assumption, and additional evidence I would request: …

You can edit a Markdown cell by double-clicking it and render it with Shift+Enter. Keep this note with your code so another reader can reconstruct your reasoning.

Use the five-minute partner comparison to read each other's notes. Ask: “Does the conclusion answer the named question? Did more than one assumption change? Is the limitation specific to this analysis?” Revise one sentence after hearing the challenge. Different defensible policies can lead to different answers.

## Reading C — How exploration can reveal or exaggerate a pattern

**Read and reflect for 10–15 minutes.** Read through Checkpoint C before running the distribution examples. Your aim is to distinguish a numerical pattern from the interpretation attached to it.

### There is more than one meaning of “typical”

Exploratory data analysis means looking at distributions, relationships, and examples to understand what might be happening. It helps us generate and refine questions. It does not automatically confirm that a pattern will recur in new data, explain its cause, or show that an intervention would work.

Suppose five invented invoices have values £10, £10, £15, £20, and £445. Their total is £500, their mean is £100, and their median is £15. The mean uses every value and answers what an equal share of the total would be. The median is the middle value after sorting: half the observations lie on either side, allowing for ties. Calling £100 “a typical purchase” could give a reader the wrong impression when four of the five purchases are £20 or less.

Neither statistic is universally better. If you need the total, the large invoice matters. If you want to describe an ordinary-sized purchase in this example, the median is more informative. Reporting both lets a reader see that the distribution is uneven. Quantiles add detail: a 90th percentile marks a value at or below which approximately 90% of observations fall, with the exact estimate depending on the quantile convention. It is not a boundary between correct and incorrect observations.

### Change the unit before summarizing purchases

Our source contains invoice lines. To describe invoice values, first group those lines by `InvoiceNo` and sum their line values. In `groupby(...).agg(...)`, the grouping field identifies which rows belong together and the aggregation defines how to combine each measurement. We sum money and units, count product lines, and retain the earliest recorded date for the invoice.

Notice that a line count differs from a count of distinct products. Repeated product codes may appear on different lines. Likewise, the number of invoices differs from the number of customers. When reading an output table, say its unit aloud: “one row per invoice.” This small habit prevents many interpretation errors.

The feature `LineValue` equals quantity times unit price. Our current view excludes returns, cancellations, and nonpositive prices. Its sum is therefore positive recorded sales value under that policy. The aggregation does not suddenly turn it into profit, net revenue, or lifetime customer value. A calculation carries its input assumptions forward.

### Reading a histogram with a long tail

A histogram divides a numerical range into intervals, called bins, and counts how many observations fall into each interval. If most invoices are small and a few are extremely large, the original-scale plot may compress almost everything against the left edge. That long tail is part of the data, not a plotting failure.

The second histogram displays $log(1+x)$ of invoice value. The logarithm compresses large differences so that the smaller values become easier to distinguish. We add one so that zero would have a defined transformed value; our positive-sales invoices are already greater than zero. This is a change of display coordinates, not a deletion of large purchases. All the invoices are still present.

Read the axis labels carefully. A horizontal coordinate of 6 in the transformed histogram is not £6. It corresponds to an original value of about $e^6-1$, or £402. You do not need to calculate this for every observation, but you do need to remember what space you are looking at. In the later scatterplot, the axes instead show original values using logarithmic spacing; equal distances along the axis represent multiplicative changes.

Changing the number of bins can reveal or hide apparent bumps. It cannot create new observations, but it changes the picture presented to the reader. If a claimed group exists only under one convenient binning choice, investigate further before describing it as a real category.

### A relationship is not an explanation

In the scatterplot, each point is an invoice. One axis shows product-line count and the other shows positive invoice value. More lines may accompany more value, but quantity and unit price also matter. One expensive product can generate more value than many inexpensive products. Inspecting specific invoices helps connect the aggregate picture to the records behind it.

Now compare countries. A large country total can arise because the extract contains many invoices from that country, because its invoices are large, or both. Dividing by the invoice count changes the question to average observed invoice value. It does not correct all selection differences or establish purchasing preferences.

Imagine country A contributes two invoices worth £10 and £190, while country B contributes twenty invoices worth £30 each. A has the larger mean (£100 versus £30), while B has the larger total (£600 versus £200). A's mean is also heavily affected by one purchase. A minimum-invoice threshold can make the comparison less dependent on very small groups, but choosing a threshold does not remove bias or prove statistical significance.

### Challenge the result before naming a story

A useful investigation asks whether its description survives a reasonable alternative: a different minimum group size, a median instead of a mean, or a sensitivity view without the largest invoices. Removing the top tail is not automatically legitimate cleaning. It answers “what would this summary look like without those observations?” Report what was excluded and retain the original result alongside it.

Time comparisons require similar care. December 2011 is incomplete in our source. A partial month can still have a high total if it contains one very large invoice. That observation does not establish growth, and dividing by the number of observed days would add assumptions about how representative those days are. Historical coverage, our sampling design, and individual large events all matter.

### Checkpoint C — rewrite an overclaim

Rewrite this sentence into a claim our analysis could support: “People in country A prefer expensive products because A has the highest average invoice value.” Name two other explanations for that average. Then explain why showing a log histogram is different from excluding the top 1% of invoices.

<details><summary>Self-check — open after writing</summary>

A defensible version states that country A has the highest mean positive invoice value among eligible countries in this extract under the named policy. It does not attribute motives to people. A few large orders, different quantities, wholesale activity, or sample selection could help explain the average. A log display retains the observations and changes their spacing; top-tail exclusion changes which observations contribute to the summary.

</details>

In [ ]:
invoices = sales.groupby('InvoiceNo').agg(
    value=('LineValue', 'sum'), units=('Quantity', 'sum'),
    lines=('StockCode', 'size'), date=('InvoiceDate', 'min'))
display(invoices[['value', 'units', 'lines']].describe(percentiles=[.5, .9, .99]).round(2))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].hist(invoices.value, bins=35, color='#277da8')
axes[0].set(xlabel='Positive invoice value (GBP)', ylabel='Invoice count', title='Original units')
axes[0].ticklabel_format(axis='x', style='sci', scilimits=(0, 0))
axes[1].hist(np.log1p(invoices.value), bins=35, color='#f8961e')
axes[1].set(xlabel='log(1 + positive invoice value)', ylabel='Invoice count', title='Same invoices, log display')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(invoices.lines, invoices.value, alpha=.35, s=16)
ax.set(xscale='log', yscale='log', xlabel='Product-line count (log scale)',
       ylabel='Positive invoice value in GBP (log scale)', title='More lines does not imply proportional value')
plt.tight_layout(); plt.show()
display(invoices.nlargest(5, 'value'))
# TODO 6: Inspect one large invoice in sales. Is it many products or high quantity?
invoice_to_inspect = invoices.value.idxmax()
display(sales[sales.InvoiceNo == invoice_to_inspect].nlargest(5, 'LineValue'))

## Independent investigation 2 — challenge a finding (40 minutes)

Run the baseline country tables below, then choose **one** route. Spend about 5 minutes choosing and predicting, 20 minutes experimenting, and 15 minutes writing an evidence report. You may reuse earlier cells, add cells, or adapt the starter cell below. Avoid editing the supplied CSV or overwriting `sales`.

**Route A: is the ranking about market size or purchase size?** Compare country rankings by total value and value per invoice. Then change `min_invoices` from 5 to 20, holding the data policy fixed. Record which countries remain eligible, any ranking changes, and each selected country's invoice count. If the ranking stays the same, report that and explain what did change. Choose a suitable chart and explain why a high average does not establish preference.

**Route B: how much does the largest tail matter?** Use the optional starter cell after the baseline plots to compare all invoices with a view excluding invoice values above a chosen quantile. Begin with 0.99 and try one other threshold. Report the number of invoices removed, their share of positive value, and changes in mean and median. Keep the original result visible. Is your new view an alternative description or a justified correction? Explain what evidence would be required to call it a correction.

For both routes, write a prediction before running. Compare at least two settings, label the question and unit on the output, and explain an observed record or group behind the aggregate. A result that contradicts your prediction is useful evidence, not a failed exercise.

<details><summary>Hints — choose only what you need</summary>

Route A: the `country` table already has `positive_value`, `invoices`, and `value_per_invoice`. Filter with `country[country.invoices >= min_invoices]`, then sort on your chosen measure. You can plot a selected column with `.plot.barh()`. Route B: `invoices.value.quantile(0.99)` gives a threshold; select invoices using that threshold, not individual product lines. To recover all lines of retained invoices, use `sales[sales.InvoiceNo.isin(retained_invoice_ids)]`. Equal values at the threshold mean the removed fraction need not be exactly 1%.

</details>

**Finished?** You have a labeled figure, a comparison table, a short report with actual numbers, and one limitation that affects your conclusion. If finished early, try the other route or compare the known-ID and all-sales country summaries below.

In [ ]:
country = sales.groupby('Country').agg(
    positive_value=('LineValue', 'sum'), invoices=('InvoiceNo', 'nunique'))
country['value_per_invoice'] = country.positive_value / country.invoices
display(country.sort_values('positive_value', ascending=False).head(8).round(2))
# TODO 7: Change this threshold from 5 to 20. Explain why the ranking may change.
min_invoices = 5
eligible = country[country.invoices >= min_invoices]
display(eligible.sort_values('value_per_invoice', ascending=False).head(8).round(2))

In [ ]:
identified, _ = sales_view(raw, require_customer=True)
comparison = pd.concat([
    sales.groupby('Country').LineValue.sum().rename('all_positive_sales'),
    identified.groupby('Country').LineValue.sum().rename('known_ID_sales')], axis=1).fillna(0)
comparison['retained_percent'] = 100 * comparison.known_ID_sales / comparison.all_positive_sales
display(comparison.sort_values('all_positive_sales', ascending=False).head(8).round(2))
monthly = sales.groupby(sales.InvoiceDate.dt.to_period('M')).LineValue.sum()
monthly.index = monthly.index.astype(str)
monthly.plot.bar(color='#277da8')
plt.ylabel('Positive recorded sales value (GBP)')
plt.title('Extract totals; final December is incomplete, sampling is selective')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

In [ ]:
# Route B starter: optional for Route A, but runs safely for everyone.
# Change one threshold, predict its effect, and keep the original baseline intact.
tail_quantile = 0.99
threshold = invoices.value.quantile(tail_quantile)
without_tail = invoices[invoices.value <= threshold].copy()
tail_comparison = pd.DataFrame({
    'all_invoices': [len(invoices), invoices.value.sum(), invoices.value.mean(), invoices.value.median()],
    'without_upper_tail': [len(without_tail), without_tail.value.sum(),
                           without_tail.value.mean(), without_tail.value.median()]},
    index=['invoice_count', 'positive_value_GBP', 'mean_invoice_GBP', 'median_invoice_GBP'])
display(tail_comparison.round(2))
removed_count = len(invoices) - len(without_tail)
removed_share = 100 * (invoices.value.sum() - without_tail.value.sum()) / invoices.value.sum()
print(f'Threshold: GBP {threshold:,.2f}; removed invoices: {removed_count}; removed value: {removed_share:.2f}%')
tail_comparison.loc[['mean_invoice_GBP', 'median_invoice_GBP']].plot.bar(rot=0)
plt.ylabel('Positive invoice value (GBP)')
plt.title('Sensitivity to upper-tail exclusion — not automatic data cleaning')
plt.tight_layout(); plt.show()


### Evidence report — write in this cell

**TODO 8:** write 150–200 words answering the prompts below. Your output is an argument supported by results, not a list of code operations.

1. **Question:** what are you describing, for which observation unit?
2. **Prediction:** what did you expect your controlled change to do, and why?
3. **Evidence:** give the baseline and changed values, with units and group sizes. Refer to one labeled figure.
4. **Interpretation:** what does the comparison suggest? Separate what you observed from a possible explanation.
5. **Limit:** name one thing the comparison cannot establish and one extra source of evidence that would help.

Useful sentence structure: “Among ___ in this extract, under policy ___, the observed ___ was ___. After changing ___, it became ___. This supports the description ___, but does not establish ___.” Replace every blank with a concrete statement; do not copy the structure without interpreting it.

### Final self-check and discussion

Without looking back, explain why a missing customer ID need not invalidate a sale, why a repeat flag does not prove an error, and why the highest country average is not proof of preference. Identify which assumption most affected your own result. These are the concepts to retain even if you forget the pandas syntax.

Next week we represent each identified customer by observed purchasing behavior. Today's choice of what counts as a purchase will shape tomorrow's distances and groups.

## Reading path

- **Start here · 15–20 min:** the [UCI dataset description](https://archive.ics.uci.edu/dataset/352/online+retail) and variable definitions. Question: which fields tell you the observation unit and cancellation convention? Compare its metadata claims with our measured audit.
- **Go deeper · 25–35 min:** [Datasheets for Datasets](https://arxiv.org/abs/1803.09010), sections on composition, collection, and preprocessing. Question: which three unanswered questions would prevent a confident business recommendation? You need not read the full paper.
- **Try it · 15 min:** [pandas: Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html), detection and dropping/filling examples. Question: why does filling a missing customer ID with one shared ID create a false customer?

## Homework / Post-class Extensions — optional

1. Compare retaining and excluding the top 1% of invoice values. Treat exclusion as a sensitivity analysis, not a correction; report both totals.
2. Build a missingness table by month and explain what the sampling design prevents you from inferring.
3. Write a one-page dataset card using the reading. Separate observed facts from assumptions.
4. Try [UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing): inspect categorical `unknown` values and determine whether `duration` is available at the intended prediction time. This alternative dataset is not required for class.